In [6]:
from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv
import pandas as pd,os,json

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")
client = Anthropic(api_key=api_key)
model = "claude-sonnet-4-6"

df = pd.read_csv(r"telehealth.csv")
temp_df = df.copy()
temp_df.fillna("N/A", inplace=True)


# def tools_data():
    
def get_df_info():
    temp_df = df.copy()
    temp_df.fillna("N/A", inplace=True)

    return {
        "columns": temp_df.columns.tolist(),
        "rows": len(temp_df),
        "data_types": temp_df.dtypes.astype(str).to_dict(),
        "top_5_rows": temp_df.head().to_dict(orient="records"),
    }

def run_cohort_analysis(df):
    result={}
    
    for program,group in df.groupby("program_type"):
        
        total_patients=len(group)
        active_patients=(
            group["retention_status"].astype(str).str.lower().isin(["active", "completed"]).sum()
        )
        
        churned_patients=total_patients-active_patients
        
        retention_rate=round((active_patients/total_patients)*100,2)
        
        result[program]={
            "number_of_patients":total_patients,
            "average_treatment_duration_weeks":round(group["treatment_duration_weeks"].mean(),2),
            "active_patients":int(active_patients),
            "churned_patients":int(churned_patients),
            "retention_rate_percent":retention_rate,
        }

    return result

def run_outcome_analysis(df):
    result={}
    
    for program,group in df.groupby("program_type"):
        retention_rate = round(
            (group["retention_status"].astype(str).str.lower().isin(["active", "completed"]).sum() / len(group)) * 100, 2
        )
        duration_q1 = round(group["treatment_duration_weeks"].quantile(0.25), 2)
        duration_median = round(group["treatment_duration_weeks"].quantile(0.5), 2)
        duration_q3 = round(group["treatment_duration_weeks"].quantile(0.75), 2)
        duration_mean = round(group["treatment_duration_weeks"].mean(), 2)
        
        churned = group[group["retention_status"].astype(str).str.lower().eq("dropped off")]
        
        early_dropoff = len(churned[churned["treatment_duration_weeks"] <= duration_q1])
        mid_dropoff = len(churned[(churned["treatment_duration_weeks"] > duration_q1) & 
                                (churned["treatment_duration_weeks"] <= duration_q3)])
        late_dropoff = len(churned[churned["treatment_duration_weeks"] > duration_q3])
        
        result[program] = {
            "retention_rate_percent": retention_rate,
            "duration_trends": {
                "mean_weeks": duration_mean,
                "median_weeks": duration_median,
                "q1_weeks": duration_q1,
                "q3_weeks": duration_q3,
            },
            "drop_off_points": {
                "early_stage_patients": int(early_dropoff),
                "mid_stage_patients": int(mid_dropoff),
                "late_stage_patients": int(late_dropoff),
            }
        }
    
    return result

def flag_anomalies(df):
    result = {}
    
    for program, group in df.groupby("program_type"):
        q1 = group["treatment_duration_weeks"].quantile(0.25)
        q3 = group["treatment_duration_weeks"].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - (1.5 * iqr)
        
        short_duration_outliers = group[group["treatment_duration_weeks"] < lower_bound]
        short_duration_count = len(short_duration_outliers)
        
        churned = group[group["retention_status"].astype(str).str.lower().eq("dropped off")]
        churn_rate = round((len(churned) / len(group)) * 100, 2)
        
        abnormal_churn = churn_rate > 25.0
        
        result[program] = {
            "short_duration_outliers": {
                "count": int(short_duration_count),
                "threshold_weeks": round(lower_bound, 2),
                "affected_patients_percent": round((short_duration_count / len(group)) * 100, 2),
                "flagged": short_duration_count > 0,
            },
            "abnormal_drop_off_clusters": {
                "churn_rate_percent": churn_rate,
                "flagged": abnormal_churn,
                "interpretation": "High churn detected" if abnormal_churn else "Normal churn levels"
            }
        }
    
    return result


tools = [
    {
        "name": "get_df_info",
        "description": "Returns raw dataset structure: column names, data types, row count, and sample rows. Use this only to inspect what data exists before running any analysis. Provides no statistics, comparisons, trends, or risk flags.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "run_cohort_analysis",
        "description": "Returns a point-in-time snapshot per program for side-by-side comparison: patient counts and retention percentage. Use for ranking or comparing programs against each other, including ranking by churn or retention. Does not analyze drop-off timing or flag statistical risk — for timing use run_outcome_analysis, for risk thresholds use flag_anomalies.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "run_outcome_analysis",
        "description": "Returns time-based behavior per program: how treatment length is distributed, and which stage (early, mid, late) churned patients left at. Use for understanding when or at what point in treatment patients drop off. Does not rank programs against each other or flag statistical risk — for ranking use run_cohort_analysis, for risk thresholds use flag_anomalies.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    },
    {
        "name": "flag_anomalies",
        "description": "Returns statistical risk flags per program: treatment durations that are extreme outliers (IQR method) and churn rates exceeding a 25% threshold. Use for risk detection or exception flagging, not raw counts or timing. Does not provide patient counts or drop-off timing — for counts use run_cohort_analysis, for timing use run_outcome_analysis.",
        "input_schema": {
            "type": "object",
            "properties": {},
            "required": []
        }
    }
]

#coordinator agent 
def coordinator_agent(temp_df):
    top_10_rows = temp_df.head(10).to_dict(orient="records")
    response = client.messages.create(
        model=model,
        max_tokens=3000,
        system=""" You are the Coordinator Agent in a multi-agent healthcare data analysis system.

You will receive ONLY the first 10 rows of a healthcare dataset.

Your sole responsibility is to generate four detailed task prompts for four independent analysis agents.

Do NOT analyze the dataset yourself.
Do NOT summarize the dataset.
Do NOT make observations or conclusions.
Do NOT mention tools or how to use them.
Only generate task prompts.

Each prompt should:
- Be tailored to the dataset based on the sample rows.
- Reference actual column names whenever appropriate.
- Clearly define the objective of the assigned analysis.
- Instruct the subagent to inspect the complete dataset before beginning its analysis.
- Encourage the subagent to support findings with statistics, comparisons, and visualizations where appropriate.
- Adapt to the available data instead of assuming specific healthcare fields exist.

Generate prompts for the following agents:

1. Program Performance Agent
Write a prompt instructing the agent to evaluate the overall performance of healthcare programs, providers, hospitals, treatments, or services (whichever is relevant). The prompt should ask the agent to identify key performance indicators, compare entities, rank performance, explain trends, and provide actionable insights.

2. Patient Segmentation Agent
Write a prompt instructing the agent to identify meaningful patient segments using the available demographic, clinical, geographic, behavioral, or utilization-related variables. If meaningful segmentation is not supported by the dataset, instruct the agent to explain why and identify the closest alternative grouping analysis.

3. Retention & Outcome Analysis Agent
Write a prompt instructing the agent to analyze patient retention, treatment completion, follow-up adherence, revisit patterns, treatment duration, and patient outcomes where applicable. If the dataset cannot support longitudinal analysis, instruct the agent to explain why and perform the closest meaningful utilization analysis instead.

4. Anomaly Detection Agent
Write a prompt instructing the agent to identify unusual patterns, statistical outliers, inconsistent records, abnormal utilization, high-risk programs, missing or suspicious data, and other potential anomalies. The prompt should ask the agent to explain the significance of each finding and prioritize issues by impact.

Return ONLY the following format:

Program Performance Prompt:
<prompt>

Patient Segmentation Prompt:
<prompt>

Retention & Outcome Analysis Prompt:
<prompt>

Anomaly Detection Prompt:
<prompt>

Do not output anything else.""",  
        messages=[
            {"role": "user", "content":"Here are the first 10 rows of the dataset: " + str(top_10_rows)}
        ]
    )
    print(response.content[0].text)
    
    
coordinator_agent(temp_df)

Program Performance Prompt:
<prompt>
You are a Program Performance Agent analyzing a healthcare dataset. Before beginning any analysis, inspect the complete dataset to understand its full scope, distribution, and any nuances in the data.

Your objective is to evaluate the overall performance of the healthcare programs available in this dataset. The dataset contains the following columns: `patient_id`, `program_type`, `start_date`, `treatment_duration_weeks`, `retention_status`, and `drop_off_reason`.

Perform the following analyses:

1. **Program Inventory & Volume**: Identify all unique values in `program_type` and calculate the total number of patients enrolled in each program. Determine which programs have the highest and lowest enrollment volumes.

2. **Key Performance Indicators (KPIs)**: For each `program_type`, calculate the following KPIs:
   - Completion rate: percentage of patients with `retention_status` = 'Completed'
   - Active rate: percentage of patients with `retention_